In [18]:
pip install pandas plotly

Note: you may need to restart the kernel to use updated packages.


In [19]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

In [20]:
world = pd.read_csv('world.csv')

In [21]:
app = dash.Dash(__name__)

In [22]:
world = world[(world["Year"] >= 2000) & (world["Year"] <= 2022)]

In [23]:
app.layout = html.Div([
    html.H1("Interactive Choropleth Map", style={'text-align': 'center'}),
    
    html.Div([
        html.Label("Select Metric:"),
        dcc.Dropdown(
            id="metric-dropdown",
            options=[
                {"label": "GDP per Capita (PPP)", "value": "GDP per capita, PPP (constant 2017 international $)"},
                {"label": "Population", "value": "Population (historical)"},
                {"label": "Global Hunger Index (2021)", "value": "Global Hunger Index (2021)"},
            ],
            value="GDP per capita, PPP (constant 2017 international $)",
            style={"width": "50%"}
        ),
    ]),

    html.Div([
        html.Label("Select Year:"),
        dcc.Slider(
            id="year-slider",
            min=2000,
            max=2022,
            value=2021,
            marks={year: str(year) for year in range(2000, 2023)},
            step=1
        ),
    ], style={"margin-top": "20px"}),

    dcc.Graph(id="choropleth-map")
])

In [24]:
@app.callback(
    Output("choropleth-map", "figure"),
    [Input("metric-dropdown", "value"),
     Input("year-slider", "value")]
)
def update_map(selected_metric, selected_year):
    filtered_data = world[world["Year"] == selected_year]
    
    fig = px.choropleth(
        filtered_data,
        locations="Code",  # ISO 3166-1 alpha-3 country codes
        color=selected_metric,  # Metric to visualize
        hover_name="Entity",  # Country names
        color_continuous_scale=px.colors.sequential.Blues,
        title=f"Global {selected_metric} ({selected_year})",
    )
    return fig

if __name__ == "__main__":
    app.run_server(debug=True)